# S0 · Preparation of daily balance histories

We validate and prepare the observed balances for customer clustering. We check data coverage, apply the agreed correction for 5 August 2022 and describe balance movements and calendar patterns. We retain every account and date through the cutoff.

We run the study with `medium.csv` or `small.csv` from `data/raw/`. We save the prepared balances, calendar profiles, event flags and provenance for Stage 1.

In [1]:
import json
import sys
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
import yaml
from IPython.display import display

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/bank_balance_forecasting").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))
pio.renderers.default = "plotly_mimetype"
pl.Config.set_tbl_rows(8)
pl.Config.set_tbl_cols(10)


def style_figure(figure, height=420):
    """Fit the output column and keep the plot readable at notebook scale."""
    figure.update_layout(
        width=None,
        autosize=True,
        height=height,
        template="plotly_white",
        font_size=12,
        title_font_size=15,
        margin=dict(l=60, r=28, t=68, b=70),
        legend=dict(orientation="h", x=0, y=-0.19, yanchor="top", font_size=11),
    )
    for menu in figure.layout.updatemenus:
        menu.update(x=1, xanchor="right", y=1.02, yanchor="bottom")
    return figure


def show(figure, height=420):
    """Render an interactive figure at the width of the notebook output."""
    style_figure(figure, height).show(config={"responsive": True, "displaylogo": False})

## 1 · Sample and forecast origin

We set the account sample with `DATASET` and limit each diagnostic to the history available through `AS_OF_DATE`. We use the same sample and cutoff in Stage 1. The numerical rules come from `parameters.yml`.

In [2]:
DATASET = "medium"  # "small" also works
AS_OF_DATE = "2022-12-31"  # use the same values in both notebooks
DATA_ROOT = ROOT / "data"
STAGE0_DIR = DATA_ROOT / "prepared" / DATASET / AS_OF_DATE
settings = yaml.safe_load((ROOT / "parameters.yml").read_text())

parameters = {
    **settings["stage0"],
    "source_path": str(DATA_ROOT / "raw" / f"{DATASET}.csv"),
    "as_of_date": AS_OF_DATE,
}
source_path = Path(parameters["source_path"])
if not source_path.is_file():
    raise FileNotFoundError(f"We could not find the source CSV at {source_path}")
display(pl.DataFrame({"dataset": [DATASET], "cutoff": [AS_OF_DATE]}))

dataset,cutoff
str,str
"""medium""","""2022-12-31"""


In [3]:
from bank_balance_forecasting.preprocessing.events import (
    classify_reversals,
    flag_extreme_changes,
    flag_level_shifts,
)
from bank_balance_forecasting.preprocessing.ingestion import (
    apply_corrections,
    classify_missing_history,
    prepare_history,
    screen_shared_dates,
)
from bank_balance_forecasting.preprocessing.profiles import (
    profile_activity,
    profile_calendar,
    profile_regimes,
)
from bank_balance_forecasting.preprocessing.reporting import report_findings

## 2 · Source validation and available history

We validate unique account IDs, a consecutive daily calendar and finite observed balances. The first preview shows the source layout with one row per account and one balance column per date. After reshaping, the second preview shows rows by account and date. We check the account count and date bounds against the study sample and record the source checksum.

In [4]:
source = pl.scan_csv(source_path, infer_schema=False)
columns = source.collect_schema().names()
display(source.select(columns[:6]).head(5).collect())
history, source_manifest = prepare_history(source, parameters)
display(history.head())
display(
    history.select(
        pl.col("account_id").n_unique().alias("accounts"),
        pl.col("date").n_unique().alias("calendar_days"),
        pl.col("date").min().alias("first_date"),
        pl.col("date").max().alias("last_date"),
        pl.col("balance_raw").null_count().alias("missing_balances"),
    )
)

accounts,calendar_days,first_date,last_date,missing_balances
u32,u32,date,date,u32
500,1095,2020-01-02,2022-12-31,5853


## 3 · Missing observations

We classify missing dates as leading, internal, trailing or part of a wholly missing history. In the heatmap, leading bands show shorter observed histories, internal gaps show interruptions and vertical bands flag dates for a shared data collection check. We preserve nulls and their dates to avoid creating artificial inactivity or movement through imputation.

In [5]:
classified_history, coverage = classify_missing_history(history.lazy())
missing_counts = classified_history.group_by("quality_flags").len().sort("quality_flags")
display(missing_counts)
display(coverage.filter(pl.col("missing_days") > 0).head())
missing = history.with_columns(pl.col("balance_raw").is_null().cast(pl.Int8).alias("missing"))
missing = missing.pivot(on="date", index="account_id", values="missing").sort("account_id")
show(
    px.imshow(
        missing.drop("account_id").to_numpy(),
        x=missing.columns[1:],
        y=missing["account_id"].to_list(),
        aspect="auto",
        zmin=0,
        zmax=1,
        color_continuous_scale=[[0, "#edf2f4"], [1, "#d1495b"]],
        labels={"x": "Date", "y": "Account", "color": "Missing"},
        title="Missing calendar observations (1 = missing, 0 = observed)",
    )
)

quality_flags,len
str,u32
"""observed""",541647
"""prehistory_missing""",5853


## 4 · Screening for shared recording anomalies

We count exact doublings followed by a return and flag review dates with the near-doubling screen. The line shows the daily share of eligible accounts with a balance near twice the following day's balance. The dotted line marks the review threshold. We investigate spikes as possible shared recording issues.

In [6]:
shared_date_flags = screen_shared_dates(history.lazy(), parameters)
display(shared_date_flags.filter(pl.col("review")))
fig = px.line(
    shared_date_flags.to_pandas(),
    x="date",
    y="share",
    title="Share of eligible accounts with a near-double pattern",
)
fig.add_hline(y=parameters["shared_date_share"], line_dash="dot")
show(fig)

date,eligible,near_double,exact_double_return,share,review
date,u32,u32,u32,f64,bool
2022-08-05,472,312,92,0.661017,true


## 5 · Balance correction on 5 August 2022

We divide each observed balance on **5 August 2022** by **2** and keep the raw value. The table lists both values and the multiplier. We use assertions to verify the factor, the missing-value count and the preservation of balances on other dates. The chart compares daily means from the same observed accounts on each date.

In [7]:
prepared_balances, correction_log = apply_corrections(classified_history.lazy(), parameters)
display(correction_log.select("account_id", "date", "balance_raw", "balance", "multiplier").head())
corrected_dates = correction_log["date"].unique().to_list()
assert correction_log.filter(pl.col("balance") != pl.col("balance_raw") * 0.5).is_empty()
assert prepared_balances["balance"].null_count() == history["balance_raw"].null_count()
assert (
    prepared_balances.filter(~pl.col("date").is_in(corrected_dates))
    .select(pl.col("balance").eq_missing(pl.col("balance_raw")).all())
    .item()
)
zoom = (
    prepared_balances.filter(pl.col("date").is_between(pl.date(2022, 8, 1), pl.date(2022, 8, 10)))
    .group_by("date")
    .agg(pl.col("balance_raw").mean(), pl.col("balance").mean())
    .sort("date")
)
if zoom.height:
    show(
        px.line(
            zoom.to_pandas(),
            x="date",
            y=["balance_raw", "balance"],
            markers=True,
            title="Mean observed balance around the correction (same accounts per date)",
        )
    )

## 6 · Unusually large account movements

We score daily movements using a robust scale estimated from the account's own nonzero changes. This puts quiet and volatile accounts on comparable scales. The status table shows where we could estimate a scale. We carry the ranked candidates into the reversal check and retain their balances in the history.

In [8]:
large_changes, change_diagnostics = flag_extreme_changes(prepared_balances.lazy(), parameters)
display(change_diagnostics.group_by("change_screen_status").len())
display(large_changes.sort("score", descending=True).head())

change_screen_status,len
str,u32
"""ok""",500


## 7 · Reversals after large movements

We examine the 30 days after each candidate movement. We count a reversal when the balance differs from its pre-movement level by at most 10% of the movement's magnitude. We count observed reversals and unresolved events. Events near the cutoff have less follow-up time. The chart marks candidates on the account history containing the highest-scored movement.

In [9]:
movement_events = classify_reversals(prepared_balances.lazy(), large_changes.lazy(), parameters)
display(movement_events.group_by("candidate_type").len())
if large_changes.height:
    example_account = large_changes.sort("score", descending=True)["account_id"][0]
    example = prepared_balances.filter(pl.col("account_id") == example_account).sort("date")
    candidates = example.join(
        movement_events.filter(pl.col("account_id") == example_account),
        on=["account_id", "date"],
        how="inner",
    )
    fig = px.line(
        example.to_pandas(),
        x="date",
        y="balance",
        title="Largest-scored movement's account: retained history",
    )
    fig.add_trace(
        go.Scatter(
            x=candidates["date"].to_list(),
            y=candidates["balance"].to_list(),
            mode="markers",
            name="Movement candidates",
        )
    )
    show(fig)

candidate_type,len
str,u32
"""temporary_excursion""",9358
"""large_change_unresolved""",10291


### Unresolved drops into near-zero balances

We review the three largest flagged drops ending within €5 of zero. For each one, we compare the prepared balance with the source and count how long it stayed at the new level. We distinguish one-day readings from persistent events using follow-up through the selected cutoff. The €5 threshold applies to this review.

In the medium sample, we found a drop from **€204,493.55 to −€2.29 on 25 November 2022**. The source and prepared values agree, and the new balance persists for 37 days through the cutoff. We retain the observed values and record `large_change_unresolved` in the event table. We would need account records to confirm the cause before applying a correction.

In [10]:
unresolved = movement_events.filter(pl.col("candidate_type") == "large_change_unresolved")
keys = ["account_id", "date"]
drops = (
    large_changes.filter((pl.col("change") < 0) & (pl.col("balance").abs() <= 5))
    .join(unresolved.select(keys), on=keys, how="semi")
    .sort("change")
    .head(3)
)
episodes = (
    prepared_balances.join(drops.select("account_id").unique(), on="account_id", how="semi")
    .sort(keys)
    .with_columns(pl.col("balance").rle_id().over("account_id").alias("run"))
    .group_by("account_id", "run")
    .agg(
        pl.col("date").min(),
        pl.col("date").max().alias("last_day"),
        pl.len().alias("days_at_new_balance"),
        (pl.col("balance_raw") == pl.col("balance")).all().alias("matches_source"),
    )
)
display(
    drops.join(episodes, on=keys, validate="1:1")
    .sort("change")
    .select(
        "date", "previous_balance", "balance", "last_day", "days_at_new_balance", "matches_source",
    )
)

## 8 · Sustained changes in balance level

We compare the median balance in adjacent 28-day windows and retain the strongest boundary from each contiguous run of candidates. This detects changes that develop across several days. We rank the boundaries in the table and use monthly counts to examine shared dates and changing behaviour. The bars show raw event counts, so the number of accounts available for screening can affect comparisons across months.

In [11]:
level_shift_flags = flag_level_shifts(prepared_balances.lazy(), parameters)
display(level_shift_flags.sort("score", descending=True).head())
if level_shift_flags.height:
    shifts_by_month = (
        level_shift_flags.group_by(pl.col("date").dt.truncate("1mo").alias("month"))
        .len()
        .sort("month")
    )
    show(
        px.bar(
            shifts_by_month.to_pandas(),
            x="month",
            y="len",
            title="Sustained level-change candidates by month",
        )
    )

## 9 · Inactivity, zero balances and negative balances

We record runs of unchanged balances, movement counts and the frequency of zero or negative balances. Missing dates break unchanged runs. The histogram shows each account's longest observed flat run. We use the totals to count zero and negative balances and accounts with zero recorded movements in the available history.

In [12]:
activity_profiles = profile_activity(prepared_balances.lazy(), parameters)
display(activity_profiles.head())
show(
    px.histogram(
        activity_profiles.to_pandas(),
        x="longest_constant_run",
        nbins=35,
        title="Longest unchanged run per account (calendar days)",
    )
)
display(
    activity_profiles.select(
        pl.col("zero_days").sum(),
        pl.col("negative_days").sum(),
        (pl.col("movement_days") == 0).sum().alias("accounts_without_observed_movements"),
    )
)

zero_days,negative_days,accounts_without_observed_movements
i64,i64,u32
6,119881,0


## 10 · Weekday differences in movement frequency

We divide movement days by valid adjacent-day pairs and pool the counts by weekday. Monday is 1 and Sunday is 7. Pooling weights accounts by the number of available pairs. We use weekday differences to assess calendar activity and retain the account-level table for Stage 1.

In [13]:
calendar_profiles = profile_calendar(prepared_balances.lazy(), parameters)
weekday_activity = (
    calendar_profiles.group_by("weekday")
    .agg((pl.col("movement_days").sum() / pl.col("eligible_pairs").sum()).alias("movement_rate"))
    .sort("weekday")
)
show(
    px.bar(
        weekday_activity.to_pandas(),
        x="weekday",
        y="movement_rate",
        title="Observed movement rate by weekday (1 = Monday, 7 = Sunday)",
    )
)

## 11 · Monthly behaviour from 2020 onward

We use the same complete-history accounts each month to control for changes in cohort membership. The count shows the size of this fixed cohort. We chart balance levels and movement frequency over the period that includes COVID-19. We interpret the comparison descriptively because the available history starts in 2020.

In [14]:
regime_profiles = profile_regimes(prepared_balances.lazy(), parameters)
complete_accounts = coverage.filter(pl.col("missing_days") == 0).select("account_id")
monthly_cohort = (
    regime_profiles.join(complete_accounts, on="account_id")
    .group_by("month")
    .agg(
        pl.col("median_balance").median().alias("median_account_balance"),
        pl.col("movement_rate").mean().alias("mean_movement_rate"),
    )
    .sort("month")
)
display(pl.DataFrame({"complete_history_accounts": [complete_accounts.height]}))

complete_history_accounts
i64
475


### Monthly balance levels

We take the median of the account-level monthly median balances to describe a typical account over time. Sustained changes motivate the regime descriptors in Stage 1. The aggregate median can conceal offsetting movements across accounts.

In [15]:
if monthly_cohort.height:
    show(
        px.line(
            monthly_cohort.to_pandas(),
            x="month",
            y="median_account_balance",
            markers=True,
            title="Fixed-cohort monthly median balances",
        )
    )
else:
    print("No complete-history cohort is available at this cutoff.")

### Monthly movement activity

We take the mean of the account-level monthly movement rates in the same cohort. We compare it with the balance-level series to see whether changes in the typical balance coincide with changes in movement frequency. The rates reflect each month's observed weekday composition.

In [16]:
if monthly_cohort.height:
    show(
        px.line(
            monthly_cohort.to_pandas(),
            x="month",
            y="mean_movement_rate",
            markers=True,
            title="Fixed-cohort monthly movement rate",
        )
    )

## 12 · Reconciliation with the source

We review counts of accounts, dates, missing values, corrections and event candidates. We verify that all source accounts and calendar rows remain. Quality flags record missingness and corrections, and the event table records movements and level shifts for Stage 1. We count identical histories for review and use `is_usable` to identify observed balances.

In [17]:
account_profiles, event_flags, quality_report, _ = report_findings(
    prepared_balances.lazy(),
    coverage.lazy(),
    activity_profiles.lazy(),
    calendar_profiles.lazy(),
    regime_profiles.lazy(),
    movement_events.lazy(),
    level_shift_flags.lazy(),
    change_diagnostics.lazy(),
    correction_log.lazy(),
    shared_date_flags.lazy(),
    source_manifest,
    parameters,
)
metrics = [
    "accounts",
    "calendar_days",
    "rows",
    "missing_values",
    "usable_observations",
    "corrected_balances",
    "event_candidates",
    "duplicate_series_groups",
]
display(pl.DataFrame({"metric": metrics, "value": [quality_report[key] for key in metrics]}))
display(prepared_balances.group_by("quality_flags").len().sort("quality_flags"))
assert prepared_balances.height == history.height
assert prepared_balances["account_id"].n_unique() == source_manifest["accounts"]

metric,value
str,i64
"""accounts""",500
"""calendar_days""",1095
"""rows""",547500
"""missing_values""",5853
"""usable_observations""",541647
"""corrected_balances""",500
"""event_candidates""",21862
"""duplicate_series_groups""",0


quality_flags,len
str,u32
"""known_date_scaled""",500
"""observed""",541147
"""prehistory_missing""",5853


## 13 · Inputs for Stage 1

We save `prepared_balances` for balance features, `calendar_profiles` for calendar features and `event_flags` for movement and regime features. We record the sample, cutoff, source checksum and settings. The file and row counts verify the handoff under `data/prepared/`, which lets Stage 1 run in its own kernel.

In [18]:
STAGE0_DIR.mkdir(parents=True, exist_ok=True)
inputs = {
    "prepared_balances": prepared_balances,
    "calendar_profiles": calendar_profiles,
    "event_flags": event_flags,
}
for name, table in inputs.items():
    table.write_parquet(STAGE0_DIR / f"{name}.parquet")
metadata = {
    "dataset": DATASET,
    "as_of_date": AS_OF_DATE,
    "source_sha256": source_manifest["sha256"],
    "stage0_parameters": {key: value for key, value in parameters.items() if key != "source_path"},
}
(STAGE0_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, default=str) + "\n")
display(
    pl.DataFrame(
        {
            "file": [f"{name}.parquet" for name in inputs],
            "rows": [table.height for table in inputs.values()],
        }
    )
)
print(f"Stage 1 inputs: {STAGE0_DIR.relative_to(ROOT)}")

file,rows
str,i64
"""prepared_balances.parquet""",547500
"""calendar_profiles.parquet""",108500
"""event_flags.parquet""",21862


Stage 1 inputs: data/prepared/medium/2022-12-31
